# Tutorial: Compare Aligned Spaces

Audience:
- Python users comparing two views over the same observations.

Prerequisites:
- Equal-length, aligned record lists and callable metrics.

Learning goals:
- Compare spaces by position with `compare`/`correlate`.
- Read `CorrelationResult` metadata.
- Handle mismatched record counts.

## Outline

1. Import the public facade.
2. Build two aligned spaces.
3. Compare by position.
4. Correlate by position.
5. Try a small exercise.

In [ ]:
from metric import Space
from metric import metrics


## Step 1 - Build source and quality spaces

Both spaces describe the same observations with different record values.

In [ ]:
ids = ["run-a", "run-b", "run-c", "run-d"]
process = Space([0, 1, 2, 3], metric=lambda lhs, rhs: abs(lhs - rhs), ids=ids)
quality = Space([0, 1, 4, 9], metric=lambda lhs, rhs: abs(lhs - rhs), ids=ids)

assert process.ids == quality.ids
len(process.pairwise())


## Step 2 - Compare by position

The promoted exact path compares records in their current order and returns a
`CorrelationResult` carrying the Pearson correlation of the two pairwise
distance profiles.

In [ ]:
comparison = process.compare(quality)

assert comparison.left_record_count == 4
assert comparison.right_record_count == 4
assert comparison.pair_count == 6
assert comparison.align == "position"
assert comparison.statistic_name == "distance_profile_correlation"
assert comparison.diagnostics["defined"]
comparison.value


## Step 3 - Correlate is the same statistic

`correlate` is an alias of `compare` for the promoted positional path, so the
two calls agree.

In [ ]:
correlation = process.correlate(quality)

assert correlation.value == comparison.value
assert correlation.align == "position"
correlation.value


## Exercise

Aligned comparison needs equal record counts. Comparing spaces of different
sizes raises `IncompatibleSpaceError` and names both counts.

Common pitfall: positional alignment compares records by their current order, so
keep both spaces in the same observation order.

In [ ]:
from metric.exceptions import IncompatibleSpaceError

shorter = Space([0, 1, 4], metric=lambda lhs, rhs: abs(lhs - rhs), ids=["run-a", "run-b", "run-c"])

try:
    process.compare(shorter)
    raise AssertionError("aligned compare should reject mismatched counts")
except IncompatibleSpaceError as exc:
    message = str(exc)

assert "4 records" in message
assert "3 records" in message
message
